1. Import library

In [18]:
import pandas as pd

2. Mengkoneksikan colab dengan g-drive

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


3. Lokasi file dataset kotor

In [20]:
file_path = '/content/drive/MyDrive/Colab Notebooks/messy_IMDB_dataset.xlsx'
data = pd.read_excel(file_path, engine='openpyxl')

4. Tampilan Data Kotor

In [21]:
print(data)

    IMBD title ID               Original titlÊ         Release year  \
0       tt0111161     The Shawshank Redemption  1995-02-10 00:00:00   
1       tt0068646                The Godfather           09 21 1972   
2       tt0468569              The Dark Knight          23 -07-2008   
3       tt0071562       The Godfather: Part II  1975-09-25 00:00:00   
4       tt0110912                 Pulp Fiction  1994-10-28 00:00:00   
..            ...                          ...                  ...   
96      tt0070735                    The Sting  1974-03-21 00:00:00   
97      tt0082096                     Das Boot  1982-03-18 00:00:00   
98      tt0059578  Per qualche dollaro in piÃ¹  1965-12-20 00:00:00   
99      tt1832382      Jodaeiye Nader az Simin  2011-10-21 00:00:00   
100     tt0045152          Singin' in the Rain  1953-02-05 00:00:00   

                         Genrë¨ Duration       Country Content Rating  \
0                         Drama      142           USA              R   
1

5. Tampilan Struktur data kotor film

In [22]:
print("Struktur Data Film")
print(data.info())

Struktur Data Film
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   IMBD title ID   100 non-null    object 
 1   Original titlÊ  100 non-null    object 
 2   Release year    100 non-null    object 
 3   Genrë¨          100 non-null    object 
 4   Duration        99 non-null     object 
 5   Country         100 non-null    object 
 6   Content Rating  77 non-null     object 
 7   Director        100 non-null    object 
 8   Unnamed: 8      0 non-null      float64
 9   Income          100 non-null    object 
 10   Votes          100 non-null    float64
 11  Score           100 non-null    object 
dtypes: float64(2), object(10)
memory usage: 9.6+ KB
None


6. Tampilan Kolom data kotor

In [23]:
column_types = data.dtypes
print(column_types)

IMBD title ID      object
Original titlÊ     object
Release year       object
Genrë¨             object
Duration           object
Country            object
Content Rating     object
Director           object
Unnamed: 8        float64
Income             object
 Votes            float64
Score              object
dtype: object


7. Membersihkan Nama Kolom dan Menghapus Kolom Kosong

In [24]:
# 1. Menghapus kolom yang sepenuhnya kosong (Unnamed: 8)
data = data.dropna(axis=1, how='all')

# 2. Membersihkan nama kolom
data.rename(columns={
    'IMBD title ID': 'Title_ID',
    'Original titlÊ': 'Original_Title',
    'Release year': 'Release_Date',
    'Genrë¨': 'Genre',
    ' Votes ': 'Votes' # Menghapus spasi di awal/akhir
}, inplace=True)

print(data.columns)

Index(['Title_ID', 'Original_Title', 'Release_Date', 'Genre', 'Duration',
       'Country', 'Content Rating', 'Director', 'Income', 'Votes', 'Score'],
      dtype='object')


8. Deduplikasi Data (Menghapus Data Duplikat yang di Kolom ID dan Judul Film)


In [25]:
# Cek jumlah data sebelum deduplikasi
print("Jumlah baris sebelum deduplikasi:", data.shape[0])

# Menghapus duplikat berdasarkan kolom Title_ID
data = data.drop_duplicates(subset=['Title_ID'], keep='first')

# Cek jumlah data setelah deduplikasi
print("Jumlah baris setelah deduplikasi:", data.shape[0])

Jumlah baris sebelum deduplikasi: 101
Jumlah baris setelah deduplikasi: 101


In [26]:
# Cek jumlah data sebelum deduplikasi
print("Jumlah baris sebelum deduplikasi:", data.shape[0])

# Menghapus duplikat berdasarkan kolom Original_Title
data = data.drop_duplicates(subset=['Original_Title'], keep='first')

# Cek jumlah data setelah deduplikasi
print("Jumlah baris setelah deduplikasi:", data.shape[0])

Jumlah baris sebelum deduplikasi: 101
Jumlah baris setelah deduplikasi: 101


9. Data Cleansing per kolom




A. Kolom Income (Pendapatan)

In [27]:
# Membersihkan karakter non-numerik pada Income
data['Income'] = data['Income'].astype(str)
data['Income'] = data['Income'].str.replace('$', '', regex=False)
data['Income'] = data['Income'].str.replace(',', '', regex=False)
data['Income'] = data['Income'].str.replace('o', '0', regex=False) # Memperbaiki typo 'o' menjadi '0'
data['Income'] = data['Income'].str.strip()

# Mengubah tipe datanya menjadi float
data['Income'] = pd.to_numeric(data['Income'], errors='coerce')

B. Kolom Score (Skor)

In [28]:
import re

# Fungsi untuk mengekstrak hanya angka dan titik
def clean_score(x):
    x = str(x).replace(',', '.') # Ubah koma jadi titik
    cleaned = re.sub(r'[^0-9.]', '', x) # Hapus semua huruf/karakter selain angka dan titik
    return cleaned

data['Score'] = data['Score'].apply(clean_score)
data['Score'] = pd.to_numeric(data['Score'], errors='coerce')

C. Kolom Release_Date (Tahun Rilis)

In [29]:
# Standarisasi ke format datetime (akan mengubah format yang aneh menjadi NaT jika tidak terbaca)
data['Release_Date'] = pd.to_datetime(data['Release_Date'], errors='coerce')

# (Opsional) Data Enrichment: Membuat kolom baru khusus 'Year'
data['Release_Year'] = data['Release_Date'].dt.year

D. Kolom Country (Negara)

In [30]:
# Menyamakan 'US' menjadi 'USA'
data['Country'] = data['Country'].replace('US', 'USA')

E. Mengisi Nilai Kosong pada Duration dan Content Rating

In [31]:
# Mengubah Duration ke numerik, memaksa error menjadi NaN
data['Duration'] = pd.to_numeric(data['Duration'], errors='coerce')

# Mengisi NaN pada Duration dengan nilai median
median_duration = data['Duration'].median()
data['Duration'] = data['Duration'].fillna(median_duration)

# Mengisi NaN pada Content Rating dengan 'Unknown'
data['Content Rating'] = data['Content Rating'].fillna('Unknown')

4. Data Enrichment (Pengayaan Data)

In [32]:
# Membuat fungsi kategorisasi skor
def categorize_score(score):
    if pd.isna(score):
        return 'Unknown'
    elif score >= 8.5:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Low'

# Menerapkan fungsi untuk membuat kolom baru
data['Score_Category'] = data['Score'].apply(categorize_score)

# Menampilkan hasil akhir
print(data.head())

    Title_ID            Original_Title Release_Date                 Genre  \
0  tt0111161  The Shawshank Redemption   1995-02-10                 Drama   
1  tt0068646             The Godfather   1972-09-21          Crime, Drama   
2  tt0468569           The Dark Knight   2008-07-23  Action, Crime, Drama   
3  tt0071562    The Godfather: Part II   1975-09-25          Crime, Drama   
4  tt0110912              Pulp Fiction   1994-10-28          Crime, Drama   

   Duration Country Content Rating              Director        Income  \
0     142.0     USA              R        Frank Darabont  2.881524e+07   
1     175.0     USA              R  Francis Ford Coppola  2.461210e+08   
2     152.0     USA          PG-13     Christopher Nolan  1.005455e+09   
3     220.0     USA              R  Francis Ford Coppola  4.080358e+08   
4     129.5     USA              R     Quentin Tarantino  2.228318e+08   

       Votes  Score  Release_Year Score_Category  
0  2278845.0    9.3        1995.0        